# 태스크 1: Strands Agents를 통한 개인 예산 도우미 구축

## 개요

이 태스크에서는 Strands Agents를 사용하여 정교한 개인 예산 도우미를 만들 것입니다. 기본 대화형 에이전트부터 시작하여 모델 구성, 대화 관리, 사용자 지정 도구, 구조화된 출력 등의 고급 기능을 추가하여 점진적으로 개선할 것입니다.

이 태스크는 실제 구현을 통해 Stand Agents의 핵심 개념을 설명하고, 각 기능이 이전 기능을 기반으로 포괄적인 재무 자문 시스템을 만드는 방법을 보여줍니다. 결과적으로 맞춤형 예산 자문, 지출 분석 및 재정 권장 사항을 제공할 수 있는 프로덕션 지원 에이전트가 생길 것입니다.

우리는 사용자가 지능적인 대화와 전문 도구를 통해 개인 재정을 관리할 수 있도록 도와주는 포괄적인 **예산 에이전트**를 만들고 있습니다. 에이전트는 예산 책정 지침을 제공하고, 지출 패턴을 분석하고, 실행 가능한 재정 자문을 제공합니다.

![아키텍처](./images/single-agent_ko_kr.png)

### 예산 에이전트 도구 및 기능

| 도구 | 설명 | 예제 사용 사례 |
|------|-------------|------------------|
| **calculate_budget** | 월 소득을 기준으로 50/30/20 예산 내역을 계산합니다. | “월 5,000달러를 벌고 있는 경우 예산을 세워줘” |
| **create_financial_chart** | 파이 차트 및 재무 데이터를 시각화합니다. | “다양한 범주에 걸친 내 지출을 시각화해줘” |
| **calculator** | 재무 계획을 위한 수학적 계산을 수행합니다. | “저축을 위해 월 소득의 15% 를 계산해 줘” |

### 에이전트 기능 요약

예산 에이전트에는 다음이 포함됩니다.

- **맞춤형 재무 지침**: 소득 및 지출 패턴을 기반으로 한 맞춤형 조언
- **대화형 예산 책정**: 검증된 50/30/20 규칙을 사용한 실시간 예산 계산
- **시각적 분석**: 더 나은 재무 데이터 이해를 위한 차트 생성
- **대화 메모리**: 여러 상호 작용에 대한 컨텍스트 보존을 통한 개인화된 경험 제공
- **구조화된 보고**: 상태 점수 및 권장 사항이 포함된 일관되고 분석 가능한 재무 보고서
- **책임감 있는 AI**: 윤리적 재정 자문을 위한 내장 가드레일 및 고지 사항

에이전트는 예산 책정 및 지출 분석에만 집중하여 투자 조언 없이 실용적이고 실행 가능한 지침을 제공합니다. 이는 후속 작업에서 구축하게 될 보다 복잡한 다중 에이전트 시스템의 기반이 됩니다.

In [ ]:
%%capture
# [환경 준비] 실습에 필요한 라이브러리를 설치한다.
#   --force-reinstall -U : 이미 설치돼 있어도 최신 버전으로 강제 재설치
#   -r requirements.txt  : 이 파일에 적힌 패키지 목록을 한 번에 설치
#   --quiet 등           : 설치 로그를 줄여 노트북을 깔끔하게 유지
# %%capture 매직: 이 셀의 출력(긴 설치 로그)을 화면에 찍지 않고 삼킨다.
!pip install --force-reinstall -U -r requirements.txt --quiet --disable-pip-version-check

In [ ]:
# [임포트] 예산 에이전트에 필요한 Strands 구성 요소와 유틸리티를 가져온다.
from strands import Agent, tool          # Agent: 에이전트 본체 / tool: 함수를 '도구'로 등록하는 데코레이터
from strands.models import BedrockModel  # Amazon Bedrock 모델을 Strands에 연결하는 어댑터
from strands_tools import calculator     # Strands가 기본 제공하는 계산기 도구
from utils import create_guardrail, pretty_print_messages  # 실습 제공 헬퍼(가드레일 생성, 메시지 보기 좋게 출력)
import boto3                              # AWS SDK for Python. 리전 조회 등에 사용
import time
import matplotlib.pyplot as plt          # 파이차트 시각화용
import logging
from typing import Union
from decimal import Decimal

# [리전 자동 감지] 코드에 리전을 하드코딩하지 않고, 현재 자격 증명의 기본 리전을 읽는다.
# 이렇게 하면 어느 리전에서 실습하든 같은 코드가 동작한다.
region = boto3.Session().region_name

# [로깅 설정] 에러 추적·디버깅용. 도구 실행 중 예외를 logger로 남긴다(아래 도구 셀에서 사용).
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

### Amazon Bedrock 가드레일을 Strands와 연관시키세요

Amazon Bedrock은 Strands Agents SDK와 직접 통합되는 [내장 가드레일 프레임워크](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html)를 제공합니다. 가드레일이 트리거되면 Stans Agents SDK가 대화 기록에 입력한 내용을 자동으로 덮어씁니다. 이는 후속 질문이 동일한 질문으로 차단되지 않도록 하기 위한 것입니다. 이를 가드레일_redact_input 부울과 가드레일_redact_input_message 문자열로 구성하여 덮어쓰기 메시지를 변경할 수 있습니다. 또한 모델의 출력에도 동일한 기능이 빌드되지만 기본적으로 비활성화되어 있습니다. 가드레일_redact_output 부울을 사용하여 이를 활성화하고 가드레일_redact_output_message 문자열로 덮어쓰기 메시지를 변경할 수 있습니다. 

In [ ]:
# [가드레일 생성] 콘텐츠 필터링·안전장치를 만든다. 반환값은 (id, arn) 튜플.
# create_guardrail 은 실습용 헬퍼(utils)이며, 내부적으로 Bedrock Guardrails API를 호출한다.
# 이 가드레일을 모델에 붙이면, 예를 들어 '투자 조언' 같은 차단 대상 요청을 걸러낸다.
guardrail_id, guardrail_arn = create_guardrail()

![가드레일](./images/guardrail_ko_kr.png)

다음은 코드에서 Bedrock 가드레일을 활용하는 방법의 예입니다.

**AWS 계정에 [Amazon Bedrock 기반 모델에 액세스](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)할 수 있는 올바른 AWS Marketplace 권한이 있는지 확인하십시오. 이 태스크는 Amazon Nova Pro 모델을 사용합니다.**

In [ ]:
# [모델 설정] Amazon Nova Pro 모델을 가드레일과 함께 구성한다.
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",  # 사용할 파운데이션 모델 ID
    region_name=region,
    temperature=0.0,                     # 0.0 = 결정적 응답. 재무 조언은 일관성이 중요하므로 낮게 둔다
    guardrail_id=guardrail_id,           # 위에서 만든 가드레일 연결
    guardrail_version="DRAFT",           # 가드레일 버전(DRAFT = 아직 게시 전 초안)
    guardrail_trace="enabled",           # 가드레일이 무엇을 왜 막았는지 추적 정보를 남긴다
)

In [ ]:
# [에이전트 생성] 아직 시스템 프롬프트도 도구도 없는 '기본' 에이전트.
# 다음 셀들에서 프롬프트·대화 관리자·도구를 하나씩 붙여가며 발전시킨다.
agent = Agent(model=bedrock_model)

In [ ]:
# [호출] 에이전트를 함수처럼 호출한다. 문자열을 넣으면 모델 응답 객체가 돌아온다.
# 이 한 줄이 곧 '사용자 메시지 -> 모델 -> 응답' 한 번의 왕복이다.
response_1 = agent("Hello! What can you do?")

In [ ]:
# [가드레일 동작 확인] 투자 조언을 요청해 본다.
# 위에서 가드레일을 붙였으므로, 이 요청은 차단되거나 걸러진 응답이 나와야 한다.
response_2 = agent("Bitcoin investment advice")

In [ ]:
# [대화 이력 확인] 에이전트가 지금까지 주고받은 메시지 전체를 보기 좋게 출력한다.
# agent.messages 에는 user/assistant 메시지가 순서대로 누적돼 있다.
# 어떤 요청이 가드레일에 걸렸는지, 모델이 어떻게 답했는지 디버깅할 때 유용하다.
pretty_print_messages(messages=agent.messages)

## 예산 에이전트 생성

### 태스크 1.1: 시스템 프롬프트 정의

In [ ]:
# [시스템 프롬프트 정의] 에이전트의 '역할·행동 규칙'을 문자열로 선언한다.
# 여기서 "투자 조언은 하지 않는다"고 못박아, 가드레일과 함께 이중으로 범위를 제한한다.
BUDGET_SYSTEM_PROMPT = """You are a helpful personal finance assistant. 
You provide general strategies for creating budgets, tips on financial discipline to achieve financial milestones, and analyze financial trends. 
You do not provide any investment advice. Keep responses concise and actionable. Always provide 2-3 specific steps the user can take. Focus on practical budgeting and spending advice.
"""

In [ ]:
# [프롬프트를 붙인 에이전트] 같은 모델에 시스템 프롬프트만 추가로 연결한다.
# 이제 이 에이전트는 '예산 도우미' 역할로 고정된다.
budget_agent_sys = Agent(
    model=bedrock_model, system_prompt=BUDGET_SYSTEM_PROMPT  # 시스템 프롬프트 연결
)

In [ ]:
# [테스트] 외식비가 과한지 물어본다. 시스템 프롬프트대로 '실행 가능한 2~3단계'를 답해야 한다.
response_3 = budget_agent_sys(
    "I spend $800/month on dining out. Is this too much for someone making $5000/month?"
)

### 태스크 1.2: 대화 관리자 추가

Stand Agents SDK에서 컨텍스트는 이해 및 추론을 위해 에이전트에게 제공되는 정보를 말합니다. 여기에는 다음이 포함됩니다.

- 사용자 메시지
- 에이전트 응답
- 도구 사용 및 결과
- 시스템 프롬프트

대화가 늘어남에 따라 다음과 같은 여러 가지 이유로 이러한 컨텍스트를 관리하는 것이 점점 더 중요해지고 있습니다.

- **토큰 제한**: 언어 모델에는 고정된 컨텍스트 창이 있습니다(처리할 수 있는 최대 토큰).
- **성능**: 컨텍스트가 클수록 처리 시간과 리소스가 더 많이 필요합니다.
- **관련성**: 오래된 메시지는 현재 대화와 관련성이 떨어질 수 있습니다.
- **일관성**: 논리적 흐름 유지 및 중요한 정보를 보존하는 것이 중요합니다.

#### 대화 관리자 유형

Strands Agents는 다양한 컨텍스트 관리 요구 사항을 처리할 수 있는 세 가지 유형의 대화 관리자를 제공합니다.

1. **[SlidingWindowConversationManager](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.conversation_manager.sliding_window_conversation_manager.SlidingWindowConversationManager)**(기본값): 고정된 수의 최근 메시지 쌍을 유지하는 슬라이딩 윈도우 전략을 구현하여 제한에 도달하면 가장 오래된 메시지 쌍을 자동으로 제거합니다. Agent 클래스에서 사용하는 기본 대화 관리자로, 최근 컨텍스트가 가장 중요한 대부분의 애플리케이션에 적합합니다.

2. **[NullConversationManager](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.conversation_manager.null_conversation_manager.NullConversationManager)**: 대화 기록을 수정하지 않는 간단한 구현입니다. 컨텍스트 제한을 초과하지 않는 짧은 대화, 디버깅 목적 또는 컨텍스트를 수동으로 관리하려는 경우에 유용합니다.

3. **[SummarizingConversationManager](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.conversation_manager.summarizing_conversation_manager.SummarizingConversationManager)**: 오래된 메시지를 단순히 버리는 대신 요약하여 지능적인 대화 컨텍스트 관리를 구현합니다. 이 접근 방식은 중요한 정보를 보존하면서도 문맥의 한계를 벗어나지 않기 때문에 역사적 맥락이 중요한 장기 대화에 적합합니다.

In [ ]:
# [대화 관리자 임포트] 대화가 길어질 때 오래된 메시지를 '요약'해 컨텍스트를 줄여주는 관리자.
# (모듈 2의 '압축(Compress)' 전략을 코드로 구현한 것)
from strands.agent.conversation_manager import SummarizingConversationManager

In [ ]:
# [대화 관리자 설정] 긴 대화를 자동으로 관리한다.
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.5,            # 컨텍스트를 줄여야 할 때 오래된 메시지의 50%를 요약으로 대체
    preserve_recent_messages=3,   # 최근 3개 메시지는 절대 요약하지 않고 원문 보존
)

In [ ]:
# [대화 관리 기능을 붙인 에이전트]
budget_agent_manager = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,
    conversation_manager=conversation_manager,  # 위에서 만든 요약 관리자 연결
    callback_handler=None,                       # None = 스트리밍 중간 이벤트를 콘솔에 자동 출력하지 않음
)

### 태스크 1.3: 스트리밍 응답(선택적 고급 기능)

지금까지 에이전트에게 `budget_agent_manager(query)`를 사용해 직접 요청을 전달하여 전체 응답이 형성된 뒤에 답을 받았습니다. 대화형 애플리케이션에서 더 나은 사용자 경험을 위해 **스트리밍**을 사용하여 응답이 생성되는 대로 표시할 수 있습니다.

#### 스트리밍 사용 시기:
- **웹 애플리케이션 및 챗봇** - 사용자는 실시간으로 응답이 표시되는 것을 볼 수 있습니다.
- **긴 응답** - 사용자는 전체 응답이 완료될 때까지 기다리지 않습니다.
- **더 나은 체감 성능** - 더 빠르고 반응성이 더 좋습니다.

#### 스트리밍을 사용하지 않는 경우:
- **일괄 처리 또는 백그라운드 작업** - 출력을 보는 사용자가 없는 경우입니다.
- **완전한 응답이 필요한 경우** - 다음 단계로 진행하기 전 완전한 응답이 필요한 경우입니다.
- **단순 스크립트** - 스트리밍이 불필요한 복잡성을 가중시키는 경우입니다.

Strands Agents SDK는 웹 서버, API 및 대화형 애플리케이션에 적합한 비동기 스트리밍을 위한 [stream_async](https://strandsagents.com/latest/documentation/docs/api-reference/agent/#strands.agent.agent.Stream_Async) 메서드를 제공합니다.

#### 비교: 일반 vs. 스트리밍 호출

일반 호출(블로킹) 과 스트리밍 호출의 차이점을 살펴보겠습니다.

In [ ]:
# [비교 실습] 같은 질문을 '일반 호출'과 '스트리밍 호출' 두 방식으로 실행해 차이를 본다.
query = "I make $5000/month and spend $800 on dining out. Is this too much?"

# === 방법 1: 일반 호출 (블로킹) — 응답이 '완성될 때까지 기다렸다가' 한 번에 받는다 ===
print("=" * 70)
print("METHOD 1: Regular Invocation (Non-Streaming)")
print("=" * 70)
print("Waiting for complete response...\n")
response = budget_agent_manager(query)  # 완료될 때까지 여기서 멈춘다
print(response)  # 전체 응답이 한꺼번에 출력된다

print("\n" + "=" * 70)
print("METHOD 2: Streaming Invocation (Real-time)")
print("=" * 70)
print("Streaming response as it's generated...\n")

# === 방법 2: 스트리밍 호출 — 생성되는 대로 조각(청크)을 받아 실시간 출력한다 ===
async def stream_response():
    """Stream the agent's response in real-time."""  # (설명) 응답을 실시간으로 스트리밍
    # stream_async 는 비동기 제너레이터. 응답 조각이 생길 때마다 event 를 하나씩 내보낸다.
    async for event in budget_agent_manager.stream_async(query):
        # 모든 event 가 텍스트는 아니다(도구 호출 등도 섞임). "data" 키가 있는 것만 화면에 찍는다.
        if "data" in event:
            # end="" : 줄바꿈 없이 이어 붙여 자연스러운 문장으로 / flush=True : 버퍼를 즉시 비워 실시간 표시
            print(event["data"], end="", flush=True)
            time.sleep(0.1)  # 스트리밍 효과를 눈으로 확인하기 위한 인위적 지연(실무에선 불필요)
    print()

# 주피터 셀에서는 async 함수를 await 로 바로 실행할 수 있다.
await stream_response()

print("\n" + "=" * 70)
print("💡 NOTICE: Streaming shows the response appearing gradually,")
print("   while regular invocation shows everything at once.")
print("=" * 70)

### 태스크 1.4: 재무 도구 추가 

In [ ]:
# [커스텀 도구 1] 50/30/20 예산 규칙을 계산하는 도구.
# @tool 데코레이터 + 타입 힌트 + docstring 이 세 가지가 모여 'LLM이 읽는 도구 명세'가 된다.
# LLM은 docstring 을 보고 "이 도구를 언제 써야 하는지" 판단한다.
@tool
def calculate_budget(monthly_income: float) -> str:
    """Calculate 50/30/20 budget breakdown for the given monthly income."""
    try:
        # 50/30/20 규칙: 필요(50%) / 원하는 소비(30%) / 저축(20%)
        needs = monthly_income * 0.50
        wants = monthly_income * 0.30
        savings = monthly_income * 0.20
        # f-string 의 :,.0f = 천단위 콤마 + 소수점 없이. 예: 5000 -> "5,000"
        return f"💰 Budget for ${monthly_income:,.0f}/month:\n• Needs: ${needs:,.0f} (50%)\n• Wants: ${wants:,.0f} (30%)\n• Savings: ${savings:,.0f} (20%)"
    except Exception as e:
        # 도구가 예외로 죽으면 에이전트 전체가 멈춘다. 그래서 에러를 잡아 '문자열로' 돌려준다.
        # (모듈 1의 '점진적 성능 저하' 원칙 — 실패해도 에이전트는 계속 동작)
        logger.error(f"Error in calculate_budget: {e}")
        return "❌ Error: Unable to calculate budget. Please provide a valid monthly income amount."

In [ ]:
# [커스텀 도구 2] 재무 데이터를 받아 파이차트를 그리는 도구.
@tool
def create_financial_chart(
    data_dict: dict, chart_title: str = "Financial Chart"
) -> str:
    """Create a pie chart visualization from financial data dictionary."""
    try:
        # [파싱] 딕셔너리를 차트가 요구하는 두 리스트로 분해한다.
        #   {"Needs": 2500, "Wants": 1500} -> labels=["Needs","Wants"], values=[2500,1500]
        labels = list(data_dict.keys())    # 항목 이름들
        values = list(data_dict.values())  # 항목 금액들
        colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", "#FF9FF3"]
        plt.figure(figsize=(8, 6))
        plt.pie(
            values,
            labels=labels,
            autopct="%1.1f%%",           # 각 조각에 퍼센트를 소수점 1자리로 표시
            colors=colors[: len(values)],  # 항목 수만큼만 색을 잘라 쓴다(색 부족/초과 방지)
            startangle=90,
        )
        plt.title(f"📊 {chart_title}", fontsize=14, fontweight="bold")
        plt.axis("equal")   # 원이 찌그러지지 않도록 가로세로 비율을 같게
        plt.tight_layout()
        plt.show()
        return f"✅ {chart_title} visualization created successfully!"
    except ImportError as e:
        # matplotlib 자체가 없을 때(설치 누락). 다른 오류와 구분해 별도로 처리한다.
        logger.error(f"Matplotlib import error: {e}")
        return "❌ Error: Chart visualization library not available."
    except Exception as e:
        logger.error(f"Error in create_financial_chart: {e}")
        return "❌ Error: Unable to create chart visualization."

In [ ]:
# [완성된 예산 에이전트] 프롬프트 + 대화 관리자 + 도구 3개를 모두 결합한다.
# tools 리스트에 넣은 도구를 LLM이 필요에 따라 스스로 골라 호출한다(도구 선택은 모델이 결정).
budget_agent = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,
    conversation_manager=conversation_manager,
    tools=[calculate_budget, create_financial_chart, calculator],  # 커스텀 2개 + 기본 계산기
    callback_handler=None,
)

In [ ]:
# [스트리밍 테스트] 도구를 갖춘 에이전트에 질문하고 응답을 실시간으로 출력한다.
# 이 과정에서 모델이 calculate_budget 같은 도구를 호출할 수도 있다.
async for event in budget_agent.stream_async(
    "I make $5000/month and spend $800 on dining out. Is this too much?"
):
    if "data" in event:        # 텍스트 청크만 골라낸다(도구 호출 이벤트는 건너뜀)
        print(event["data"], end="")
        time.sleep(0.1)  # 스트리밍 효과 확인용 지연

### 태스크 1.5: 재무 보고서에 구조화된 결과 추가

**구조화된 출력이란 무엇입니까?**

구조화된 출력은 AI 에이전트가 애플리케이션이 안정적으로 파싱하고 사용할 수 있는 일관되고 예측 가능한 형식으로 데이터를 반환하도록 합니다. 구조가 다른 자유 형식 텍스트 응답을 받는 대신 사전 정의된 스키마를 준수하는 데이터를 얻을 수 있습니다.

**Pydantic을 사용하는 이유는 무엇입니까?**

[Pydantic](https://docs.pydantic.dev/)은 데이터 유효성 검사 및 스키마 정의를 위한 Python 라이브러리입니다. 퍼블릭 액세스 차단을 사용하여 수행할 수 있는 작업은 다음과 같습니다.
- 예상하는 데이터의 정확한 구조 정의(필드, 유형, 제약 조건)
- 데이터가 스키마와 일치하는지 자동으로 검증
- 데이터가 일치하지 않을 때 유용한 오류 메시지 받기
- 설명 및 제약 조건 추가(예: ‘점수는 1~10점 사이여야 함’)

**Strands에서 Pydantic을 사용하는 방법:**

Strands Agent SDK는 다음과 같은 `structured_output()` 방법을 제공합니다.
1. Pydantic 모델을 스키마 정의로 사용합니다.
2. LLM에 정확한 구조와 일치하는 출력을 생성하도록 지시합니다.
3. LLM의 응답을 검증하고 Python 객체로 파싱합니다.
4. 코드에서 사용할 수 있는 형식이 안전한 객체를 반환합니다.

이는 계산, 보고 및 다른 시스템과의 통합을 위해 일관된 데이터 형식이 필요한 금융 애플리케이션에 특히 유용합니다.

In [ ]:
# [구조화 출력 준비] Pydantic 임포트.
# Pydantic 은 데이터 검증 라이브러리로, '응답이 반드시 이런 모양이어야 한다'는 스키마를 정의한다.
# 자유 텍스트 대신 정해진 필드를 가진 객체로 응답을 받고 싶을 때 쓴다.
from pydantic import BaseModel, Field
from typing import List

In [ ]:
# [스키마 정의] 재무 보고서의 '정확한 구조'를 Pydantic 모델로 선언한다.
# LLM은 이 스키마에 맞춰 응답을 생성하도록 지시받는다 -> 파싱 없이 바로 객체로 받는다.
class BudgetCategory(BaseModel):
    """Represents a single budget category with amount and percentage."""  # (설명) 예산 항목 하나 = 이름+금액+비율
    # Field(description=...) 의 설명은 LLM에게 '이 필드에 무엇을 넣어야 하는지' 알려주는 힌트가 된다.
    name: str = Field(description="Budget category name (e.g., 'Housing', 'Food')")
    amount: float = Field(description="Dollar amount allocated to this category")
    percentage: float = Field(description="Percentage of total income (0-100)")

class FinancialReport(BaseModel):
    """Complete financial report with income, budget breakdown, and recommendations.

    This Pydantic model defines the schema for structured output from our agent.
    The LLM will be instructed to generate a response that matches this exact structure.
    """
    # (설명) 소득 + 예산 분해 + 추천을 담는 완전한 재무 보고서 스키마
    monthly_income: float = Field(description="Total monthly income in dollars")
    # List[BudgetCategory] : 위에서 정의한 항목이 '여러 개' 들어가는 리스트
    budget_categories: List[BudgetCategory] = Field(
        description="List of budget categories with amounts and percentages"
    )
    recommendations: List[str] = Field(
        default_factory=list,   # 값이 없으면 빈 리스트로 시작(None 방지)
        description="List of specific, actionable financial recommendations"
    )
    financial_health_score: int = Field(
        default=5,
        ge=1,   # ge=1, le=10 : 1~10 범위를 벗어난 값은 검증 단계에서 거부된다
        le=10,
        description="Overall financial health score from 1 (poor) to 10 (excellent)"
    )

In [ ]:
# [구조화 출력 호출] structured_output 은 응답을 자유 텍스트가 아니라
# 위 FinancialReport 스키마에 '맞춰서' 생성하고, 검증된 파이썬 객체로 돌려준다.
# 프롬프트에서 필요한 필드를 명시적으로 요구해 누락을 줄인다.
structured_response = budget_agent.structured_output(
    output_model=FinancialReport,  # 이 스키마에 맞춰 응답을 만들라는 지시
    prompt=(
        "Generate a comprehensive financial report for someone earning $6000/month "
        "with $800 dining expenses. You MUST include ALL of the following in your response: "
        "1) monthly_income, 2) budget_categories (list with name, amount, percentage), "
        "3) recommendations (list of 2-3 actionable tips), "
        "4) financial_health_score (integer from 1-10)."
    ),
)

In [ ]:
# [결과 출력] structured_response 는 이제 '검증된 객체'다. 딕셔너리 파싱 없이 속성으로 바로 접근한다.
print(f"Income: ${structured_response.monthly_income:,.0f}")

# [반복문] budget_categories 리스트를 순회하며 항목별로 한 줄씩 출력한다.
for category in structured_response.budget_categories:
    # 각 category 는 BudgetCategory 객체 -> .name / .amount / .percentage 로 접근
    print(f"• {category.name}: ${category.amount:,.0f} ({category.percentage:.1f}%)")

print(f"\nFinancial Health Score: {structured_response.financial_health_score}/10")

# recommendations 가 비어 있을 수 있으므로 먼저 확인한다.
if structured_response.recommendations:
    print("\nRecommendations:")
    # enumerate(..., 1) : 인덱스를 1부터 매겨 "1. 2. 3." 번호를 붙인다
    for i, rec in enumerate(structured_response.recommendations, 1):
        print(f"{i}. {rec}")
else:
    print("\nNo specific recommendations generated.")

In [ ]:
%%writefile budget_agent.py
# ============================================================================
# [파일 익스포트] %%writefile 매직은 이 셀의 '코드'를 budget_agent.py 파일로 저장한다
# (셀을 실행해도 코드가 돌지 않고, 파일로 '쓰기'만 한다).
# 노트북에서 완성한 에이전트를 재사용 가능한 .py 모듈로 떼어내는 과정이다.
# Task 2･Task 3 에서 이 파일을 import 해서 오케스트레이터에 붙인다.
# 아래 코드는 앞 셀들에서 하나씩 만든 것(모델･프롬프트･도구･스키마)을 한 파일로 합친 것이다.
# ============================================================================
# Export complete budget agent implementation to Python file
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
from pydantic import BaseModel, Field
from typing import List, Union
from decimal import Decimal
import matplotlib.pyplot as plt
import logging
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

# Configure logging for error tracking and debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


# Define structured output models for financial data
class BudgetCategory(BaseModel):
    name: str = Field(description="Budget category name")
    amount: float = Field(description="Dollar amount for this category")
    percentage: float = Field(description="Percentage of total income")

class FinancialReport(BaseModel):
    """Complete financial report with income, budget breakdown, and recommendations.

    This Pydantic model defines the schema for structured output from our agent.
    The LLM will be instructed to generate a response that matches this exact structure.
    """
    monthly_income: float = Field(description="Total monthly income in dollars")
    budget_categories: List[BudgetCategory] = Field(
        description="List of budget categories with amounts and percentages"
    )
    recommendations: List[str] = Field(
        default_factory=list,
        description="List of specific, actionable financial recommendations"
    )
    financial_health_score: int = Field(
        default=5,
        ge=1,
        le=10,
        description="Overall financial health score from 1 (poor) to 10 (excellent)"
    )

# Enhanced system prompt for structured outputs
BUDGET_SYSTEM_PROMPT = """You are a helpful personal finance assistant. 
You provide general strategies for creating budgets, tips on financial discipline to achieve financial milestones, and analyze financial trends. You do not provide any investment advice. 

When generating financial reports, always provide:
1. Clear budget breakdowns using the 50/30/20 rule or custom allocations
2. Specific, actionable recommendations (2-3 steps)
3. A financial health score based on spending patterns
4. Practical budgeting and spending advice

Use structured output when requested to provide comprehensive financial reports."""

# Continue with previous configurations
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
)


@tool
def calculate_budget(monthly_income: float) -> str:
    """Calculate 50/30/20 budget breakdown for the given monthly income."""
    try:
        # Perform calculations
        needs = monthly_income * 0.50
        wants = monthly_income * 0.30
        savings = monthly_income * 0.20
        
        return f"💰 Budget for ${monthly_income:,.0f}/month:\n• Needs: ${needs:,.0f} (50%)\n• Wants: ${wants:,.0f} (30%)\n• Savings: ${savings:,.0f} (20%)"
    
    except Exception as e:
        logger.error(f"Error in calculate_budget: {e}")
        return "❌ Error: Unable to calculate budget. Please provide a valid monthly income amount."


@tool
def create_financial_chart(
    data_dict: dict, chart_title: str = "Financial Chart"
) -> str:
    """Create a pie chart visualization from financial data dictionary."""
    try:
        # Basic validation
        if not data_dict:
            return "❌ Error: No data provided for chart."
        
        labels = list(data_dict.keys())
        values = [float(v) for v in data_dict.values()]
        colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FECA57", "#FF9FF3"]
        
        # Create chart
        plt.figure(figsize=(8, 6))
        plt.pie(
            values,
            labels=labels,
            autopct="%1.1f%%",
            colors=colors[:len(values)],
            startangle=90,
        )
        plt.title(f"📊 {chart_title}", fontsize=14, fontweight="bold")
        plt.axis("equal")
        plt.tight_layout()
        plt.show()
        
        return f"✅ {chart_title} visualization created successfully!"
    
    except Exception as e:
        logger.error(f"Error in create_financial_chart: {e}")
        return "❌ Error: Unable to create chart visualization."


# Create our complete financial agent
budget_agent = Agent(
    model=bedrock_model,
    system_prompt=BUDGET_SYSTEM_PROMPT,
    tools=[calculate_budget, create_financial_chart, calculator],
    callback_handler=None,
)

if __name__ == "__main__":
    # Test structured output using structured_output_async
    print("\nStructured financial report:")
    structured_response = budget_agent.structured_output(
        output_model=FinancialReport,
        prompt=(
            "Generate a comprehensive financial report for someone earning $6000/month "
            "with $800 dining expenses. You MUST include ALL of the following in your response: "
            "1) monthly_income, 2) budget_categories (list with name, amount, percentage), "
            "3) recommendations (list of 2-3 actionable tips), "
            "4) financial_health_score (integer from 1-10)."
        ),
    )
    print(f"Income: ${structured_response.monthly_income:,.0f}")
    for category in structured_response.budget_categories:
        print(
            f"• {category.name}: ${category.amount:,.0f} ({category.percentage:.1f}%)"
        )
    print(f"\nFinancial Health Score: {structured_response.financial_health_score}/10")
    print("\nRecommendations:")
    for i, rec in enumerate(structured_response.recommendations, 1):
        print(f"{i}. {rec}")

In [ ]:
# [익스포트 검증] 앞 셀에서 budget_agent.py 로 저장한 파일을 실제로 실행해 본다.
# 노트북 밖에서도 같은 에이전트가 동작하는지 확인하는 단계.
# 이 파일은 Task 2에서 'from budget_agent import ...' 로 재사용된다.
!python budget_agent.py

**태스크 완료**: 전문 페르소나, 모델 구성, 대화 관리, 도구 통합 및 구조화된 출력과 같은 주요 에이전틱 AI 개념을 보여주는 정교한 예산 도우미를 성공적으로 만들었습니다.

### 다음 단계

이 노트북을 완료했습니다. 실습의 다음 부분으로 넘어가려면 실습 지침으로 돌아가서 **태스크 2**를 계속하십시오.